In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv('Weather_Traffic.csv')
# Drop the unnamed index column from the bad to_csv() call (no index=False)
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

print("Shape:", df.shape)
print(df.dtypes)
df.head()

Shape: (23802, 14)
date                  str
hour                int64
temperature_2m    float64
snowfall          float64
rain              float64
weather_code      float64
snow_depth        float64
showers           float64
Boro                  str
Vol                   str
street                str
fromSt                str
toSt                  str
Direction             str
dtype: object


,date,hour,temperature_2m,snowfall,rain,weather_code,snow_depth,showers,Boro,Vol,street,fromSt,toSt,Direction
0,2024-01-06,0,32.7335,0.0,0.0,0.0,0.0,0.0,Queens,61,BROADWAY,Queens Boulevard,Queens Boulevard Line,SB
1,2024-01-06,0,32.7335,0.0,0.0,0.0,0.0,0.0,Queens,5,GRAND AVENUE,72 Street,Mazeau Street,EB
2,2024-01-06,0,32.7335,0.0,0.0,0.0,0.0,0.0,Queens,32,BROADWAY,Queens Boulevard Line,Queens Boulevard,NB
3,2024-01-06,0,32.7335,0.0,0.0,0.0,0.0,0.0,Brooklyn,9,METROPOLITAN AVENUE,Dead end,Varick Avenue,WB
4,2024-01-06,0,32.7335,0.0,0.0,0.0,0.0,0.0,Manhattan,5,9 AVENUE,West 219 Street,West 220 Street,SB


In [15]:
# Parse timestamp
df['timestamp'] = pd.to_datetime(df['date']) + pd.to_timedelta(df['hour'], unit='h')

# Keep what was asked for + grouping keys
KEEP = [
    'timestamp',
    'Boro',
    'street',
    'Direction',
    'temperature_2m',
    'rain',
    'snowfall',
    'snow_depth',
    'showers',
    'weather_code',
    'Vol'                  # traffic volume
]
df = df[KEEP]

# Rename for clarity and consistency
df = df.rename(columns={
    'Boro':          'borough',
    'street':        'street',
    'Direction':     'direction',
    'temperature_2m':'temperature',
    'rain':          'rain',
    'snowfall':      'snowfall',
    'snow_depth':    'snow_depth',
    'showers':       'showers',
    'weather_code':  'weather_code',
    'Vol':           'traffic_volume'
})

# Check and drop rows where critical fields are null
print("Null counts:\n", df.isnull().sum())
df = df.dropna(subset=['timestamp', 'traffic_volume', 'temperature', 'rain', 'snowfall'])

print(f"\nClean shape: {df.shape}")
df.head()

Null counts:
 timestamp         0
borough           0
street            0
direction         0
temperature       0
rain              0
snowfall          0
snow_depth        0
showers           0
weather_code      0
traffic_volume    0
dtype: int64

Clean shape: (23802, 11)


,timestamp,borough,street,direction,temperature,rain,snowfall,snow_depth,showers,weather_code,traffic_volume
0,2024-01-06,Queens,BROADWAY,SB,32.7335,0.0,0.0,0.0,0.0,0.0,61
1,2024-01-06,Queens,GRAND AVENUE,EB,32.7335,0.0,0.0,0.0,0.0,0.0,5
2,2024-01-06,Queens,BROADWAY,NB,32.7335,0.0,0.0,0.0,0.0,0.0,32
3,2024-01-06,Brooklyn,METROPOLITAN AVENUE,WB,32.7335,0.0,0.0,0.0,0.0,0.0,9
4,2024-01-06,Manhattan,9 AVENUE,SB,32.7335,0.0,0.0,0.0,0.0,0.0,5


In [16]:
# Rush hour flag, AM rush: 7-9, PM rush: 16-19 (4pm-7pm)
df['hour'] = df['timestamp'].dt.hour

df['rush_hour'] = df['hour'].apply(
    lambda h: 1 if (7 <= h <= 9) or (16 <= h <= 19) else 0
)

# Heavy rain flag
# NOAA defines heavy rain as > 0.30 inches/hour
df['heavy_rain'] = (df['rain'] > 0.30).astype(int)

# Temperature ranges, grouping into buckets for NYC context
def classify_temp(temp_f):
    if temp_f <= 32:
        return 'freezing'       # at or below freezing - ice risk
    elif temp_f <= 45:
        return 'cold'           # cold, possible black ice
    elif temp_f <= 60:
        return 'cool'           # mild
    elif temp_f <= 75:
        return 'comfortable'    # ideal driving conditions
    else:
        return 'hot'            # heat, possible road effects

df['temp_range'] = df['temperature'].apply(classify_temp)

# Any precipitation flag (rain OR snow OR showers)
df['any_precipitation'] = (
    (df['rain'] > 0) | 
    (df['snowfall'] > 0) | 
    (df['showers'] > 0)
).astype(int)

# Road ID, street + direction as a unique road segment identifier
df['road_id'] = df['street'] + '_' + df['direction']

print("New columns added:")
print(df[['timestamp', 'rush_hour', 'heavy_rain', 'temp_range', 
          'any_precipitation', 'road_id']].head(10))

New columns added:
            timestamp  rush_hour  heavy_rain temp_range  any_precipitation  \
0 2024-01-06 00:00:00          0           0       cold                  0   
1 2024-01-06 00:00:00          0           0       cold                  0   
2 2024-01-06 00:00:00          0           0       cold                  0   
3 2024-01-06 00:00:00          0           0       cold                  0   
4 2024-01-06 00:00:00          0           0       cold                  0   
5 2024-01-06 01:00:00          0           0   freezing                  0   
6 2024-01-06 01:00:00          0           0   freezing                  0   
7 2024-01-06 01:00:00          0           0   freezing                  0   
8 2024-01-06 01:00:00          0           0   freezing                  0   
9 2024-01-06 01:00:00          0           0   freezing                  0   

                  road_id  
0             BROADWAY_SB  
1         GRAND AVENUE_EB  
2             BROADWAY_NB  
3  METROPO

In [17]:
# Group by same street + direction + hourly + borough
grouped_road = df.groupby(
    ['timestamp', 'borough', 'road_id', 'street', 'direction',
     'rush_hour', 'heavy_rain', 'temp_range', 'any_precipitation',
     'hour']
).agg(
    traffic_volume  = ('traffic_volume', 'sum'),   # total volume for that segment/hour
    temperature     = ('temperature',    'mean'),
    rain            = ('rain',           'mean'),
    snowfall        = ('snowfall',       'mean'),
    snow_depth      = ('snow_depth',     'mean'),
    showers         = ('showers',        'mean'),
    weather_code    = ('weather_code',   'first'),  # weather code same for all in hour
).reset_index()

print("Road-level grouped shape:", grouped_road.shape)
grouped_road.head()

Road-level grouped shape: (23707, 17)


,timestamp,borough,road_id,street,direction,rush_hour,heavy_rain,temp_range,any_precipitation,hour,traffic_volume,temperature,rain,snowfall,snow_depth,showers,weather_code
0,2024-01-06,Brooklyn,METROPOLITAN AVENUE_WB,METROPOLITAN AVENUE,WB,0,0,cold,0,0,9,32.7335,0.0,0.0,0.0,0.0,0.0
1,2024-01-06,Manhattan,9 AVENUE_SB,9 AVENUE,SB,0,0,cold,0,0,5,32.7335,0.0,0.0,0.0,0.0,0.0
2,2024-01-06,Queens,BROADWAY_NB,BROADWAY,NB,0,0,cold,0,0,32,32.7335,0.0,0.0,0.0,0.0,0.0
3,2024-01-06,Queens,BROADWAY_SB,BROADWAY,SB,0,0,cold,0,0,61,32.7335,0.0,0.0,0.0,0.0,0.0
4,2024-01-06,Queens,GRAND AVENUE_EB,GRAND AVENUE,EB,0,0,cold,0,0,5,32.7335,0.0,0.0,0.0,0.0,0.0


In [18]:
# Borough + hour grouping, one row per borough per hour
grouped_borough = df.groupby(
    ['timestamp', 'borough', 'hour',
     'rush_hour', 'heavy_rain', 'temp_range', 'any_precipitation']
).agg(
    traffic_volume  = ('traffic_volume', 'sum'),
    temperature     = ('temperature',    'mean'),
    rain            = ('rain',           'mean'),
    snowfall        = ('snowfall',       'mean'),
    snow_depth      = ('snow_depth',     'mean'),
    showers         = ('showers',        'mean'),
).reset_index()

print("Borough-level grouped shape:", grouped_borough.shape)
grouped_borough.head()

Borough-level grouped shape: (9411, 13)


,timestamp,borough,hour,rush_hour,heavy_rain,temp_range,any_precipitation,traffic_volume,temperature,rain,snowfall,snow_depth,showers
0,2024-01-06 00:00:00,Brooklyn,0,0,0,cold,0,9,32.7335,0.0,0.0,0.0,0.0
1,2024-01-06 00:00:00,Manhattan,0,0,0,cold,0,5,32.7335,0.0,0.0,0.0,0.0
2,2024-01-06 00:00:00,Queens,0,0,0,cold,0,61532,32.7335,0.0,0.0,0.0,0.0
3,2024-01-06 01:00:00,Brooklyn,1,0,0,freezing,0,10,31.2935,0.0,0.0,0.0,0.0
4,2024-01-06 01:00:00,Manhattan,1,0,0,freezing,0,3,31.2935,0.0,0.0,0.0,0.0


In [19]:
# Dataframe: Timestamp | Rain | Temperature | Snowfall | Traffic Volume | Road ID / Borough
# Road-level version (with road_id)
teammate_road_df = grouped_road[[
    'timestamp',
    'rain',
    'temperature',
    'snowfall',
    'traffic_volume',
    'road_id',
    'borough',
    'rush_hour',
    'heavy_rain',
    'temp_range',
    'any_precipitation'
]].copy()

# Borough-level version (simpler, borough as geographic grouping)
teammate_borough_df = grouped_borough[[
    'timestamp',
    'rain',
    'temperature',
    'snowfall',
    'traffic_volume',
    'borough',
    'rush_hour',
    'heavy_rain',
    'temp_range',
    'any_precipitation'
]].copy()

print("Road-level shape:", teammate_road_df.shape)
print("Borough-level shape:", teammate_borough_df.shape)
teammate_road_df.head()

Road-level shape: (23707, 11)
Borough-level shape: (9411, 10)


,timestamp,rain,temperature,snowfall,traffic_volume,road_id,borough,rush_hour,heavy_rain,temp_range,any_precipitation
0,2024-01-06,0.0,32.7335,0.0,9,METROPOLITAN AVENUE_WB,Brooklyn,0,0,cold,0
1,2024-01-06,0.0,32.7335,0.0,5,9 AVENUE_SB,Manhattan,0,0,cold,0
2,2024-01-06,0.0,32.7335,0.0,32,BROADWAY_NB,Queens,0,0,cold,0
3,2024-01-06,0.0,32.7335,0.0,61,BROADWAY_SB,Queens,0,0,cold,0
4,2024-01-06,0.0,32.7335,0.0,5,GRAND AVENUE_EB,Queens,0,0,cold,0


In [20]:
# Validation checks
print("Null counts (road-level):")
print(teammate_road_df.isnull().sum())

print("\nBorough distribution:")
print(teammate_road_df['borough'].value_counts())

print("\nRush hour distribution:")
print(teammate_road_df['rush_hour'].value_counts())

print("\nTemp range distribution:")
print(teammate_road_df['temp_range'].value_counts())

print("\nHeavy rain distribution:")
print(teammate_road_df['heavy_rain'].value_counts())

print("\nTraffic volume stats:")
print(teammate_road_df['traffic_volume'].describe())

print("\nDate range covered:")
print(f"  From: {teammate_road_df['timestamp'].min()}")
print(f"  To:   {teammate_road_df['timestamp'].max()}")

Null counts (road-level):
timestamp            0
rain                 0
temperature          0
snowfall             0
traffic_volume       0
road_id              0
borough              0
rush_hour            0
heavy_rain           0
temp_range           0
any_precipitation    0
dtype: int64

Borough distribution:
borough
Brooklyn         8414
Queens           6771
Manhattan        5445
Bronx            2981
Staten Island      96
Name: count, dtype: int64

Rush hour distribution:
rush_hour
0    16792
1     6915
Name: count, dtype: int64

Temp range distribution:
temp_range
cool           7907
comfortable    6273
cold           5523
hot            2353
freezing       1651
Name: count, dtype: int64

Heavy rain distribution:
heavy_rain
0    23696
1       11
Name: count, dtype: int64

Traffic volume stats:
count     23707
unique      789
top           4
freq        327
Name: traffic_volume, dtype: object

Date range covered:
  From: 2024-01-06 00:00:00
  To:   2024-12-11 23:00:00


In [21]:
# Main output
teammate_road_df.to_csv('clean_grouped_data.csv', index=False)

# borough-level for geographic analysis
teammate_borough_df.to_csv('clean_grouped_borough.csv', index=False)

print("   clean_grouped_data.csv    — road-level (for modeling)")
print("   clean_grouped_borough.csv — borough-level (for geographic viz)")

   clean_grouped_data.csv    — road-level (for modeling)
   clean_grouped_borough.csv — borough-level (for geographic viz)
